# Nullability and Missing Data



```{contents}
:local:
:depth: 2
```


This supporting reading extends Chapter 6's basic missing-input checks after you have studied object references and composition. Nullable annotations describe whether data may be missing; they do not validate an object at runtime. Apply these ideas to required constructor data, optional relationships, and tests of missing-data behavior.

Read the value-type and reference-type sections with References, Identity, and Copying; revisit conditional member access with Composition. The operator table is a reference, and the pattern-matching and LINQ practice are extension material. This page does not add a fourth required class meeting.

```{index} Nullable Value Types
```



```{index} Give Missing Data Its Own Meaning
```

## Give Missing Data Its Own Meaning


An unrecorded score, a recorded zero and an empty label are different states. Replacing every missing value with zero may make a screen easy to print while making its calculation wrong. Connect nullability to Chapter 11's object contracts: choose what may be absent, what absence means, and where a required value is enforced.

For a value type, the schematic declaration `ValueType? name = null;` creates a nullable value that can hold either an ordinary value or no value. `int?` is shorthand for Nullable<int>. Read `HasValue` before an explicit `.Value` access; accessing Value when HasValue is false throws. `GetValueOrDefault()` supplies the type's default, which may erase the distinction your business rule needs.

For a reference type, `string?` annotates a reference that may be null. Under nullable analysis, `string` expresses an intended non-null contract to the compiler. These annotations guide warnings; they do not install runtime validation. Required constructor input still needs a guard, and the null-forgiving `!` operator only suppresses a warning.


In [1]:
using System;
#nullable enable
static string Describe(int? score)
{
    if (!score.HasValue) return "Not recorded";
    return $"Recorded {score.Value}";
}
Console.WriteLine(Describe(null));
Console.WriteLine(Describe(0));
string? absent = null;
string empty = "";
Console.WriteLine($"Absent: {absent is null}; empty length: {empty.Length}");
Console.WriteLine($"Absent length: {absent?.Length ?? -1}");


Not recorded
Recorded 0
Absent: True; empty length: 0
Absent length: -1


The output preserves absence separately from zero. Empty is a real string object with Length zero; absent has no object whose Length can be read. `absent?.Length` produces a nullable length, and `?? -1` supplies a display fallback only when that result is missing.

| Expression or policy | Missing reference | Empty string | Nonempty string |
| --- | --- | --- | --- |
| `text?.Length` | null | 0 | Actual length |
| `text?.Length ?? -1` | -1 | 0 | Actual length |
| `text ?? "Unknown"` | Unknown | Empty string retained | Original text |

Read `?.` as conditional access, not a request to manufacture a default object. In `order?.ShippingAddress?.City ?? "Unknown"`, a missing order or address stops that access chain, then the final fallback provides a display value. An empty City is retained because empty is not null. If blank cities are also unacceptable, validate that separately.


```{index} Guide a Required-Value Guard and an Optional Label
```

## Guide a Required-Value Guard and an Optional Label


Suppose a customer must have a name but may omit a delivery note. The required name belongs in a constructor or boundary guard. The note can remain null in the model; its display fallback belongs where the display is produced. Do not use `note!` to make a missing note behave like a real string: it has no such runtime effect.

Guided prediction: what should the following print for null, empty and “Leave at desk”? The empty note has been supplied explicitly; this policy preserves it. Then change the policy to treat whitespace-only notes as absent and explain which condition must be added.


In [1]:
using System;
#nullable enable
static string NoteLabel(string? note) => note ?? "No delivery note";
Console.WriteLine($"[{NoteLabel(null)}]");
Console.WriteLine($"[{NoteLabel("")}]");
Console.WriteLine($"[{NoteLabel("Leave at desk")}]");


[No delivery note]
[]
[Leave at desk]


Use `string.IsNullOrWhiteSpace(note)` if the revised domain treats null, empty and whitespace-only strings alike. That is a business normalization decision, not the definition of null. As a second variation, replace a required-name `!` with an explicit argument guard and explain the difference between an early, informative rejection and a later NullReferenceException. The operator table below remains a quick reference. Pattern matching and LINQ exercises are optional extensions; neither is required to apply these missing-data contracts.


```{index} Nullable Value Types
```

## Nullable Value Types




By default, value types (`int`, `double`, `bool`) cannot be `null`. Append `?` to opt in:

```csharp
int  x = null;   // compile error
int? y = null;   // OK
```

Check before use with `.HasValue` / `.Value`, or use the null operators below.

In [ ]:
int? score = null;

if (score.HasValue)
    Console.WriteLine($"Score: {score.Value}");
else
    Console.WriteLine("No score yet.");

// Shorthand with null-coalescing
int display = score ?? 0;
Console.WriteLine($"Display: {display}");  // Display: 0

The output is `No score yet.` and `Display: 0`. `int?` is shorthand for `Nullable<int>`, a small wrapper with `HasValue` and `Value`. Reading `score.Value` while `HasValue` is false throws `InvalidOperationException`, so test first or use `??` to supply a default as the second line does. Arithmetic is lifted: `score + 1` is `null` when `score` is `null`, not an error, which can hide a missing value in a total if nobody checks.


```{index} Nullable Reference Types (C# 8+)
```



```{index} Nullable Reference Types (C# 8+)
```

## Nullable Reference Types (C# 8+)




Enable with `#nullable enable` (or project-wide via `<Nullable>enable</Nullable>`).
Once enabled, `string` means *non-nullable*; `string?` means *may be null*.
The compiler warns if you dereference a potentially-null reference.

In [ ]:
#nullable enable

string  name = "Alice";   // cannot be null
string? nick = null;      // may be null

// Console.WriteLine(nick.Length);  // warning: dereference of possibly null
Console.WriteLine(nick?.Length);  // OK — prints nothing if null

The output is a single empty line: `nick?.Length` evaluates to `null` because `nick` is `null`, and `Console.WriteLine` prints nothing for it. Uncomment the previous line and the compiler issues a warning, not an error; the program still compiles and would throw `NullReferenceException` at that line. Treat the warning as the bug report it is. The `!` operator silences it without fixing anything, which is why Chapter 11 reserves it for cases where the code has already proven the value is present.


```{index} The Null Operators
```



```{index} The Null Operators
```

## The Null Operators




| Operator | Name | Meaning |
|---|---|---|
| `x ?? y` | Null-coalescing | Return `x` if not null, else `y` |
| `x ??= y` | Null-coalescing assignment | Assign `y` to `x` only if `x` is null |
| `x?.Member` | Null-conditional | Access `Member` only if `x` is not null |
| `x?[i]` | Null-conditional index | Index only if `x` is not null |
| `x!` | Null-forgiving | Suppress the nullable warning (use sparingly) |

In [ ]:
// ?? — null-coalescing
string? input = null;
string result = input ?? "(none)";
Console.WriteLine(result);  // (none)

// ??= — assign only if null
input ??= "default";
Console.WriteLine(input);  // default

// ?. — null-conditional chain
string[]? words = null;
int? len = words?.Length;
Console.WriteLine(len);  // (nothing printed — len is null)

The output is `(none)`, `default`, and an empty line. `??` chose the fallback because `input` was `null`; `??=` then assigned `"default"` into `input` because it was still `null`, and would have left an existing value alone. `words?.Length` short-circuits the whole expression to `null`, so `len` is `int?`, not `int`: the result of a null-conditional access is always nullable, even when the member itself is not.


```{index} Chaining ?. and ??
```



```{index} Chaining ?. and ??
```

### Chaining `?.` and `??`




Combine them for safe, concise access with a fallback.
The next example uses the positional record classes introduced in {ref}`record-data-types`. Each declaration generates a constructor and properties; `Address?` marks an optional reference to that record object. Here `order` is null, so the chain stops before accessing either property.


In [ ]:
Order? order = null;

// Safe deep access with fallback
string city = order?.ShippingAddress?.City ?? "Unknown";
Console.WriteLine(city);  // Unknown

record Order(string? CustomerName, Address? ShippingAddress);

record Address(string City, string Zip);

```{index} Pattern Matching and Null
```



```{index} Pattern Matching and Null
```

## Pattern Matching and Null




A `switch` expression can test for `null` as its first case, then match on properties of a non-null value, with the catch-all `_` last. The compiler knows that `s` is not null in the later arms:

```csharp
string? s = GetValue();

string result = s switch
{
    null              => "(none)",
    { Length: 0 }    => "(empty)",
    _                 => s
};
```

In [ ]:
using System;

string?[] samples = { null, "", "Ana Diaz" };
foreach (string? s in samples)
{
    string result = s switch
    {
        null => "(none)",
        { Length: 0 } => "(empty)",
        _ => s
    };
    Console.WriteLine(result);
}


The output is `(none)`, `(empty)`, and `Ana Diaz`. The `{ Length: 0 }` arm is a property pattern: it only runs for a non-null string whose `Length` is zero, so there is no risk of dereferencing `null` there. Reorder the arms so `_` comes first and the compiler reports that the other arms are unreachable, a useful check that the cases are listed from most to least specific.


```{index} Practice
```

## Practice




1. Write `int SafeParse(string? input)` that returns the parsed integer, or `0` if `input` is null or not a valid number. Use `int.TryParse`.
2. Given `List<string?> names`, use LINQ and `??` to produce a `List<string>` where every null is replaced with `"Anonymous"`.
3. Rewrite this safely using null operators:
   ```csharp
   if (user != null && user.Address != null)
       Console.WriteLine(user.Address.City);
   ```

```{rubric} Footnotes
```
